<a href="https://colab.research.google.com/github/Valbu11/Neurovault-Brain-Data-Analysis/blob/main/notebooks/02_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Notebook 02 — Data Cleaning

**Goal:** Clean and transform the raw NeuroVault data so it is ready for analysis and visualization.

**What you will learn:**
- How to handle missing values (nulls)
- How to fix data types (dates, numbers, strings)
- How to select only useful columns
- How to create new columns from existing ones
- How to save clean data for the next notebooks

**Input:** `data/raw/collections.csv` + `data/raw/images.csv`  
**Output:** `data/processed/collections_clean.csv` + `data/processed/images_clean.csv`

---
## 1. Import libraries

In [10]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 50)

print('Libraries imported ✅')

Libraries imported ✅


---
## 2. Load raw data

In [11]:
# Montar Drive
from google.colab import drive
drive.mount('/content/drive')

# Cargar datos raw
df_collections = pd.read_csv('/content/drive/MyDrive/neurovault/raw/collections.csv', low_memory=False)
df_images = pd.read_csv('/content/drive/MyDrive/neurovault/raw/images.csv', low_memory=False)

print(f'Collections: {df_collections.shape[0]:,} rows × {df_collections.shape[1]} columns')
print(f'Images:      {df_images.shape[0]:,} rows × {df_images.shape[1]} columns')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Collections: 17,216 rows × 107 columns
Images:      2,000 rows × 66 columns


---
## 3. Select useful columns

Collections has 107 columns — most are empty or irrelevant. We keep only the ones that tell us something meaningful.

In [12]:
COLLECTION_COLS = [
    'id',
    'name',
    'description',
    'number_of_images',
    'add_date',
    'modify_date',
    'DOI',
    'authors',
    'paper_url',
    'owner',
    'full_dataset_url',
    'subjects',
    'scanner_make',
    'scanner_model',
    'field_strength',
]

IMAGE_COLS = [
    'id',
    'name',
    'modality',
    'map_type',
    'add_date',
    'modify_date',
    'collection',
    'description',
    'number_of_subjects',
    'brain_coverage',
    'is_valid',
]

# Keep only columns that exist in the dataframe
col_cols = [c for c in COLLECTION_COLS if c in df_collections.columns]
img_cols = [c for c in IMAGE_COLS if c in df_images.columns]

df_c = df_collections[col_cols].copy()
df_i = df_images[img_cols].copy()

print(f'Collections: {df_c.shape[1]} columns selected (from 107)')
print(f'Images:      {df_i.shape[1]} columns selected (from 66)')
df_c.head(3)

Collections: 14 columns selected (from 107)
Images:      11 columns selected (from 66)


,id,name,description,number_of_images,add_date,modify_date,DOI,authors,paper_url,owner,full_dataset_url,scanner_make,scanner_model,field_strength
0,862,AkiNikolaidis's temporary collection,NaN,0,2015-10-03T01:54:43.816421+02:00,2015-10-03T01:54:43.816501+02:00,NaN,NaN,NaN,574,NaN,NaN,NaN,NaN
1,38,AFNI_data6,Some class data used in AFNI bootcamp class,0,2014-06-04T04:15:05.515327+02:00,2014-06-04T04:15:05.515388+02:00,NaN,NaN,NaN,64,NaN,NaN,NaN,NaN
2,55,Naturally occurring future-focused thought pre...,NaN,0,2014-06-18T17:40:20.041953+02:00,2014-06-18T17:40:20.042015+02:00,NaN,NaN,NaN,70,NaN,NaN,NaN,NaN


---
## 4. Fix data types — dates

Dates come as strings. We convert them to datetime so we can extract year, month, etc.

In [13]:
# Convert dates with UTC to avoid mixed timezone errors
for df, name in [(df_c, 'collections'), (df_i, 'images')]:
    for col in ['add_date', 'modify_date']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
            print(f'  {name}.{col} → {df[col].dtype}')

print('\nDates fixed ✅')

  collections.add_date → datetime64[ns, UTC]
  collections.modify_date → datetime64[ns, UTC]
  images.add_date → datetime64[ns, UTC]
  images.modify_date → datetime64[ns, UTC]

Dates fixed ✅


---
## 5. Create new columns

From the dates we extract `year` and `month` — very useful for trend analysis.

In [14]:
# Collections
df_c['year']  = df_c['add_date'].dt.year
df_c['month'] = df_c['add_date'].dt.month
df_c['has_doi'] = df_c['DOI'].notna().astype(int)       # 1 if has DOI, 0 if not
df_c['has_authors'] = df_c['authors'].notna().astype(int)

# Images
df_i['year']  = df_i['add_date'].dt.year
df_i['month'] = df_i['add_date'].dt.month

print('New columns created:')
print('  collections → year, month, has_doi, has_authors')
print('  images      → year, month')
print()
print(df_c[['id', 'name', 'year', 'month', 'has_doi', 'has_authors']].head(5))

New columns created:
  collections → year, month, has_doi, has_authors
  images      → year, month

    id                                               name  year  month  \
0  862               AkiNikolaidis's temporary collection  2015     10   
1   38                                         AFNI_data6  2014      6   
2   55  Naturally occurring future-focused thought pre...  2014      6   
3   72              sylvia.morelli's temporary collection  2014      7   
4   75                    davaughn's temporary collection  2014      7   

   has_doi  has_authors  
0        0            0  
1        0            0  
2        0            0  
3        0            0  
4        0            0  


---
## 6. Handle missing values

We don't delete rows with nulls — we fill them with meaningful defaults.

In [15]:
# Collections — fill text nulls with 'Unknown'
text_cols = ['description', 'DOI', 'authors', 'paper_url', 'scanner_make', 'scanner_model']
for col in text_cols:
    if col in df_c.columns:
        df_c[col] = df_c[col].fillna('Unknown')

# Images — fill modality nulls with 'Unknown'
df_i['modality'] = df_i['modality'].fillna('Unknown')
df_i['map_type'] = df_i['map_type'].fillna('Unknown')

print('Null values after cleaning:')
print('Collections:')
remaining = df_c.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else '  No nulls ✅')
print('\nImages:')
remaining_i = df_i.isnull().sum()
print(remaining_i[remaining_i > 0] if remaining_i.sum() > 0 else '  No nulls ✅')

Null values after cleaning:
Collections:
full_dataset_url     7910
field_strength      16874
dtype: int64

Images:
description            44
number_of_subjects    447
brain_coverage         14
dtype: int64


---
## 7. Quick EDA — first insights

A fast look at the clean data before saving.

In [16]:
print('=== STUDIES PER YEAR ===')
print(df_c.groupby('year')['id'].count().to_string())

print('\n=== IMAGE MODALITY ===')
print(df_i['modality'].value_counts().to_string())

print('\n=== MAP TYPES ===')
print(df_i['map_type'].value_counts().head(8).to_string())

print('\n=== STUDIES WITH DOI ===')
doi_pct = df_c['has_doi'].mean() * 100
print(f'  With DOI:    {df_c["has_doi"].sum():,} ({doi_pct:.1f}%)')
print(f'  Without DOI: {(~df_c["has_doi"].astype(bool)).sum():,} ({100-doi_pct:.1f}%)')

=== STUDIES PER YEAR ===
year
2013      11
2014      84
2015     334
2016     484
2017     618
2018     913
2019     971
2020    2560
2021    2239
2022     689
2023    2197
2024    2665
2025    2621
2026     830

=== IMAGE MODALITY ===
modality
fMRI-BOLD         1582
Unknown            388
Structural MRI      13
Other                6
Diffusion MRI        5
fMRI-CBV             2
fMRI-CBF             2
EEG                  1
MEG                  1

=== MAP TYPES ===
map_type
univariate-beta map    1002
T map                   559
other                   194
Z map                   160
F map                    29
parcellation             26
Unknown                  14
ROI/mask                  6

=== STUDIES WITH DOI ===
  With DOI:    939 (5.5%)
  Without DOI: 16,277 (94.5%)


---
## 8. Save clean data

In [17]:
os.makedirs('data/processed', exist_ok=True)

df_c.to_csv('/content/drive/MyDrive/neurovault/processed/collections_clean.csv', index=False)
df_i.to_csv('/content/drive/MyDrive/neurovault/processed/images_clean.csv', index=False)

print('Clean data saved ✅')
print(f'  data/processed/collections_clean.csv → {df_c.shape[0]:,} rows × {df_c.shape[1]} columns')
print(f'  data/processed/images_clean.csv      → {df_i.shape[0]:,} rows × {df_i.shape[1]} columns')
print()
print('Next step → 03_sql_analysis.ipynb 🚀')

Clean data saved ✅
  data/processed/collections_clean.csv → 17,216 rows × 18 columns
  data/processed/images_clean.csv      → 2,000 rows × 13 columns

Next step → 03_sql_analysis.ipynb 🚀


---
## ✅ What we accomplished

- Reduced columns from 107 → 15 (collections) and 66 → 11 (images)
- Fixed date types and extracted year/month
- Created useful indicator columns: `has_doi`, `has_authors`
- Filled missing values with meaningful defaults
- Saved clean datasets ready for SQL and visualization

**Next notebook:** `03_sql_analysis.ipynb` — SQL queries with SQLite to answer business questions.